# SS3 Part B — Applying the Final Model to STADIOEquities' Real Client Data

This notebook applies the model selected as best in SS2 (Random Forest*) to STADIOEquities'
real client data extract (five tables, 2,104 clients, extracted 2026-09-07), covering registration,
onboarding behaviour, marketing engagement, and first-deposit outcomes.

Five sections:

1. Dataset loading
2. Dataset preprocessing
3. Model training
4. Model validation
5. Results visualisation

**How to run this notebook:** clone the repository, run `pip install -r requirements.txt`, place
the five STADIOEquities extract files in `ss3_data/raw/` (same filenames as supplied:
`table1_client_registration.csv`, `table2_app_web_onboarding.csv`, `table3_email_nudge_engagement.csv`,
`table4_account_kyc_funding.csv`), then run every cell top to bottom.

## Setup

This cell finds the repository root by walking up from the notebook's folder until it finds `README.md`, then builds every path from there.

In [2]:
import os
import json
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, roc_auc_score, precision_score, recall_score, f1_score,
    classification_report, RocCurveDisplay, ConfusionMatrixDisplay
)
import joblib

ROOT = Path.cwd()
while not (ROOT / "README.md").exists():
    if ROOT == ROOT.parent:
        raise FileNotFoundError(
            "Could not find README.md above the current folder. "
            "Make sure this notebook is saved inside the cloned repository."
        )
    ROOT = ROOT.parent

RAW_DIR = ROOT / "ss3_data" / "raw"
PROCESSED_DIR = ROOT / "ss3_data" / "processed"
RESULTS_DIR = ROOT / "ss3_data" / "results"
MODELS_DIR = ROOT / "models"

for d in (RAW_DIR, PROCESSED_DIR, RESULTS_DIR, MODELS_DIR):
    d.mkdir(parents=True, exist_ok=True)

print("Repository root detected at:", ROOT)
print("Raw data folder:", RAW_DIR)


Repository root detected at: C:\Users\THLAPEDI\Downloads
Raw data folder: C:\Users\THLAPEDI\Downloads\ss3_data\raw


---
## 1. Dataset loading 

STADIOEquities' extract is five tables joined on `Client ID`, covering clients registered between
September 2024 and August 2026:

- `table1_client_registration.csv` — one row per client: demographics, acquisition (2,110 rows, a
  handful of exact duplicates).
- `table2_app_web_onboarding.csv` — one row per app/web session, with onboarding progress.
- `table3_email_nudge_engagement.csv` — one row per marketing message sent.
- `table4_account_kyc_funding.csv` — **target table**: whether the client made a first deposit.
- `table5_timing_quality.csv` — record level timing/quality metadata (documents the observation
  window and missingness; not client level, so it is not merged into the model features below).

**Target:** `First Deposit Status` in table 4 — whether the client deposited within the 30-day
activation window.


In [4]:
RAW_FILES = {
    "t1": "table1_client_registration.csv",
    "t2": "table2_app_web_onboarding.csv",
    "t3": "table3_email_nudge_engagement.csv",
    "t4": "table4_account_kyc_funding.csv",
}

missing = [name for name in RAW_FILES.values() if not (RAW_DIR / name).exists()]
if missing:
    print("Missing from", RAW_DIR, ":", missing)
    print("Place the STADIOEquities extract files there before running the rest of this notebook.")
    t1 = t2 = t3 = t4 = None
else:
    t1 = pd.read_csv(RAW_DIR / RAW_FILES["t1"])
    t2 = pd.read_csv(RAW_DIR / RAW_FILES["t2"])
    t3 = pd.read_csv(RAW_DIR / RAW_FILES["t3"])
    t4 = pd.read_csv(RAW_DIR / RAW_FILES["t4"])
    print("table1 (clients):", t1.shape)
    print("table2 (sessions):", t2.shape)
    print("table3 (messages):", t3.shape)
    print("table4 (target):", t4.shape)


table1 (clients): (2110, 11)
table2 (sessions): (7730, 16)
table3 (messages): (11584, 9)
table4 (target): (2104, 6)


**Dataset loading summary:**

- Number of instances: 2,110 raw client rows in table 1 (2,104 unique clients after removing
  6 exact duplicate rows — see Section 2).
- Target column and classes: `First Deposit Status` in table 4, boolean — `True` (made a first
  deposit within the 30-day window) / `False` (did not).
- Class balance: **55% did not deposit / 45% did** .The classes are close to balanced.


In [5]:
if t4 is not None:
    print(t4["First Deposit Status"].value_counts())
    print(t4["First Deposit Status"].value_counts(normalize=True).round(3))


First Deposit Status
False    1161
True      943
Name: count, dtype: int64
First Deposit Status
False    0.552
True     0.448
Name: proportion, dtype: float64
